### BCSE432E Reinforcement Learning Lab: L29+L30
### Lab Assignment 4

### Monte Carlo Prediction

### Name: Yashvanth Karunakaran (23BAI1589)

Notebook Link: [https://github.com/Yashvanthk05/RL/blob/main/23BAI1589_LAB-4.ipynb](https://github.com/Yashvanthk05/RL/blob/main/23BAI1589_LAB-4.ipynb)

### TASK - 1: Small MDP
### AIM:
To simulate a simple decision-making system where an agent moves between states using a fixed strategy (always choosing “Right”), and to estimate how good each state is by computing the expected future rewards from that state using the Monte Carlo method

### Problem:
Agent starts in a environment with 3 states and repeatedly follows the same action. Some actions give rewards, while others give penalties. The goal is to understand Which states are more beneficial, How much reward the agent can expect if it starts from each state

### MDP

In [8]:
states = ["S1", "S2", "S3"]
actions = ["Left", "Right"]
gamma = 0.9

### Transition Function

In [9]:
def transition(state, action):
    if state == "S1":
        if action == "Right":
            return "S2", 0
        else:
            return "S1", -1

    elif state == "S2":
        if action == "Right":
            return "S3", 1
        else:
            return "S1", 0

    elif state == "S3":
        return "S3", 0

### Fixed Policy (Always Right)

In [10]:
def policy(state):
    return "Right"

### Generate Episode

In [11]:
def generate_episode():
    episode = []
    state = "S1"

    while True:
        action = policy(state)
        next_state, reward = transition(state, action)

        episode.append((state, reward))

        if state == "S3":
            break

        state = next_state

    return episode

### Monte Carlo Prediction  for 1000 episodes

In [12]:
from collections import defaultdict
import numpy as np

returns = defaultdict(list)
V = defaultdict(float)

num_episodes = 1000

for _ in range(num_episodes):
    episode = generate_episode()

    G = 0
    visited = set()

    for t in reversed(range(len(episode))):
        state, reward = episode[t]
        G = gamma * G + reward

        if state not in visited:
            returns[state].append(G)
            V[state] = np.mean(returns[state])
            visited.add(state)

### Estimated value function for each state

In [13]:
print("Estimated Value Function:")
for s in states:
    print(f"{s}: {V[s]:.4f}")

Estimated Value Function:
S1: 0.9000
S2: 1.0000
S3: 0.0000


### Inference
Monte Carlo estimation converges to stable values for all states after sufficient episodes.
Under the fixed policy (always choosing Right):
S3 has value nearly 0 since it is a terminal-like state with no future rewards.
S2 has a positive value because it directly leads to a reward of +1 before termination.
S1 also gains a positive value, as following the policy eventually leads to S2 to S3 and collects reward

### Task 2: 4x4 Gridworld
### AIM:
To model a grid-based environment where an agent moves randomly with uncertainty in movement, and to estimate the value of each grid cell (state) using the Monte Carlo method based on long-term rewards

### Problem Description:
Agent starts at the top-left corner of a 4×4 grid and tries to reach the bottom-right corner (goal). However Movements are not fully reliable (only 80% accurate), Hitting walls gives a penalty, Reaching the goal gives a reward

### Grid Setup

In [14]:
grid_size = 4
states = list(range(16))
actions = ["Up", "Down", "Left", "Right"]

gamma = 0.95
start_state = 0
terminal_state = 15

### Functions

In [15]:
def to_row_col(state):
    return divmod(state, grid_size)

def to_state(row, col):
    return row * grid_size + col

### Transition Function

In [16]:
def move(state, action):
    row,col=to_row_col(state)
    if action=="Up":
        row-=1
    elif action=="Down":
        row+=1
    elif action=="Left":
        col-=1
    elif action=="Right":
        col+=1

    if row<0 or row>=grid_size or col<0 or col>=grid_size:
        return state, -1

    next_state = to_state(row,col)

    if next_state == terminal_state:
        return next_state, 10

    return next_state, 0

### Stochastic Transition

In [17]:
import random

def step(state, action):
    if random.random() < 0.8:
        chosen_action = action
    else:
        other_actions = [a for a in actions if a != action]
        chosen_action = random.choice(other_actions)

    return move(state, chosen_action)

### Random Policy

In [18]:
def policy(state):
    return random.choice(actions)

### Generate Episode

In [19]:
def generate_episode():
    episode = []
    state = start_state

    while True:
        action = policy(state)
        next_state, reward = step(state, action)

        episode.append((state, reward))

        if state == terminal_state:
            break

        state = next_state

    return episode

### Monte Carlo Prediction

In [20]:
returns = defaultdict(list)
V = defaultdict(float)

num_episodes = 10000

for _ in range(num_episodes):
    episode = generate_episode()

    G = 0
    visited = set()

    for t in reversed(range(len(episode))):
        state, reward = episode[t]
        G = gamma * G + reward

        if state not in visited:
            returns[state].append(G)
            V[state] = np.mean(returns[state])
            visited.add(state)

### Value Function as Grid

In [21]:
print("Estimated Value Function (4x4 Grid):")

for i in range(grid_size):
    row_values = []
    for j in range(grid_size):
        s = to_state(i, j)
        row_values.append(f"{V[s]:6.2f}")
    print(" ".join(row_values))

Estimated Value Function (4x4 Grid):
  1.29   2.13   2.81   2.81
  2.08   3.33   4.45   4.70
  2.70   4.36   6.19   7.33
  2.74   4.73   7.41  -0.50


### Inference:
Value function shows a clear gradient across the grid such that
States closer to the goal (S15) have higher values.
States farther away have lower values due to penalties and longer paths.
The stochastic nature of movement (80% correct, 20% random) reduces overall state values because:
The agent may deviate from optimal paths.
It may hit boundaries and incur penalties.

### Conclusion:
Monte Carlo prediction effectively evaluates a policy even in uncertain and stochastic environments